# 01 · EDA — Product distribution (part 1)

A **1% random sample** of `to_share/inputs.csv`, with a derived `product` column.

**Ground rules for this notebook**

- The raw extract is **never modified** — it is read-only. `inputs.csv` is scanned by DuckDB, not loaded into pandas.
- The `product` column is derived using the **same logic as the production build**
  (`to_share/build_dashboard_data.py`, the `sku_product` CTE), so anything found here
  carries over to the dashboard.
- Sampling is **reproducible** — same `SAMPLE_SEED` gives the same rows every run.
- Rows are returned **ordered by `order_date`**.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

# Resolve the repo root whether the kernel starts in the repo root or in to_share/
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "to_share" / "inputs.csv").exists()
)
SALES_CSV = ROOT / "to_share" / "inputs.csv"

SAMPLE_PERCENT = 1.0   # 1% of rows
SAMPLE_SEED = 42       # change this to draw a different sample

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

con = duckdb.connect()
con.execute("PRAGMA threads=4")

print(f"source : {SALES_CSV}")
print(f"size   : {SALES_CSV.stat().st_size / 1e6:,.0f} MB")

source : /Users/deepanvishal.thulasivel/Library/CloudStorage/OneDrive-COLORIMAGEINC/Git Local/Product_level_forecast/Product_level_forecast/to_share/inputs.csv
size   : 919 MB


## 1. Draw the 1% sample

`USING SAMPLE ... (bernoulli, seed)` makes an independent coin-flip per row, so every
row has an equal chance of selection and the draw is repeatable. DuckDB streams the CSV
from disk — the full 877 MB is never materialised.

In [2]:
total_rows = con.execute(
    "SELECT count(*) FROM read_csv_auto(?, sample_size=200000)", [str(SALES_CSV)]
).fetchone()[0]

print(f"full extract : {total_rows:,} rows")
print(f"expected 1%  : ~{total_rows // 100:,} rows")

full extract : 3,485,426 rows
expected 1%  : ~34,854 rows


## 2. Derive `product` — identical logic to the production build

Lifted from `build_dashboard_data.py` (the `prepared` CTE). Two derivations:

**`product`** — the title with its trailing colour suffix removed:

| order | condition | result |
|---|---|---|
| 1 | title is NULL | `'Unknown SKU <sku>'` |
| 2 | title ends with `" - <color>"` | chop exactly `len(color) + 3` chars off the right, then `rtrim` |
| 3 | title contains any `" - "` | strip from the **first** separator onward |
| 4 | otherwise | title unchanged |

**`resolved_color`** — the inverse: use `color` when present, else pull it back out of the title.

The build normalises text (`trim`, empty-string → NULL) *before* applying this. We do the
same **inside the derivation only** — the original `product_title` and `color` columns are
returned untouched, so you can always see what the source actually said.

In [3]:
# --- verbatim from build_dashboard_data.py, sku_product CTE ---------------------
TITLE_NORM = "nullif(trim(CAST(product_title AS VARCHAR)), '')"
COLOR_NORM = "nullif(trim(CAST(color AS VARCHAR)), '')"

PRODUCT_SQL = rf"""
    CASE
        WHEN {TITLE_NORM} IS NULL THEN concat('Unknown SKU ', sku)
        WHEN {COLOR_NORM} IS NOT NULL
             AND ends_with(lower({TITLE_NORM}), lower(concat(' - ', {COLOR_NORM})))
            THEN rtrim(left({TITLE_NORM},
                            length({TITLE_NORM}) - length({COLOR_NORM}) - 3))
        WHEN regexp_matches({TITLE_NORM}, '\s+-\s+')
            THEN regexp_replace({TITLE_NORM}, '\s+-\s+.*$', '')
        ELSE {TITLE_NORM}
    END
"""

RESOLVED_COLOR_SQL = rf"""
    coalesce({COLOR_NORM},
             nullif(regexp_extract({TITLE_NORM}, '\s+-\s+(.+)$', 1), ''))
"""

print(PRODUCT_SQL)


    CASE
        WHEN nullif(trim(CAST(product_title AS VARCHAR)), '') IS NULL THEN concat('Unknown SKU ', sku)
        WHEN nullif(trim(CAST(color AS VARCHAR)), '') IS NOT NULL
             AND ends_with(lower(nullif(trim(CAST(product_title AS VARCHAR)), '')), lower(concat(' - ', nullif(trim(CAST(color AS VARCHAR)), ''))))
            THEN rtrim(left(nullif(trim(CAST(product_title AS VARCHAR)), ''),
                            length(nullif(trim(CAST(product_title AS VARCHAR)), '')) - length(nullif(trim(CAST(color AS VARCHAR)), '')) - 3))
        WHEN regexp_matches(nullif(trim(CAST(product_title AS VARCHAR)), ''), '\s+-\s+')
            THEN regexp_replace(nullif(trim(CAST(product_title AS VARCHAR)), ''), '\s+-\s+.*$', '')
        ELSE nullif(trim(CAST(product_title AS VARCHAR)), '')
    END



## 3. Build the sample, ordered by date

`product` and `resolved_color` are **added** columns. Every source column is passed through
as-is via `r.*`.

In [4]:
query = f"""
WITH raw AS (
    SELECT *
    FROM read_csv_auto('{SALES_CSV}', sample_size=200000)
    USING SAMPLE {SAMPLE_PERCENT} PERCENT (bernoulli, {SAMPLE_SEED})
)
SELECT
    r.*,
    {PRODUCT_SQL}        AS product,
    {RESOLVED_COLOR_SQL} AS resolved_color
FROM raw r
ORDER BY r.order_date, r.product_title, r.sku
"""

sample = con.execute(query).df()
print(f"sampled : {len(sample):,} rows  ({len(sample) / total_rows:.3%} of source)")
print(f"dates   : {sample['order_date'].min()}  ->  {sample['order_date'].max()}")

sampled : 34,715 rows  (0.996% of source)
dates   : 2025-01-01 00:00:00  ->  2026-09-15 00:00:00


## 4. What the table looks like

In [5]:
print(f"shape: {sample.shape[0]:,} rows x {sample.shape[1]} columns\n")
pd.DataFrame({
    "dtype": sample.dtypes.astype(str),
    "non_null": sample.notna().sum(),
    "nulls": sample.isna().sum(),
    "distinct": [sample[c].nunique(dropna=True) for c in sample.columns],
})

shape: 34,715 rows x 24 columns



,dtype,non_null,nulls,distinct
order_date,datetime64[us],34715,0,623
product_title,object,34715,0,5093
sku,object,34715,0,15181
product_id,Int64,33990,725,6844
price,float64,33990,725,282
color_group,object,33771,944,14
color,object,33809,906,405
item_class,object,33990,725,66
subcategory,object,33990,725,63
category,object,33990,725,25


In [6]:
# First rows, ordered by date
sample[[
    "order_date", "product_title", "product", "color", "resolved_color",
    "sku", "revenue", "ordered_quantities",
]].head(15)

,order_date,product_title,product,color,resolved_color,sku,revenue,ordered_quantities
0,2025-01-01,"7"" Sports Club Sweater Knit Basketball Short -...","7"" Sports Club Sweater Knit Basketball Short",Black,Black,M6150R012,148.00,1
1,2025-01-01,7/8 High-Waist Airbrush Legging - Classic Red,7/8 High-Waist Airbrush Legging,Classic Red,Classic Red,W5604R046801,172.00,2
2,2025-01-01,Accolade 1/4 Zip Pullover - Athletic Heather Grey,Accolade 1/4 Zip Pullover,Athletic Grey,Athletic Grey,U3040RG029104,2273.38,17
3,2025-01-01,Accolade Crew Neck Pullover - Celestial Blue,Accolade Crew Neck Pullover,Celestial Blue,Celestial Blue,U3031RG058166,256.00,2
4,2025-01-01,Accolade Full Zip Hoodie - Black,Accolade Full Zip Hoodie,Black,Black,U3041RG014,3086.69,23
5,2025-01-01,Accolade Hoodie - Athletic Heather Grey,Accolade Hoodie,None,Athletic Heather Grey,U3032RG029100,356.17,3
6,2025-01-01,Accolade Hoodie - Bone,Accolade Hoodie,None,Bone,U3032RG030400,414.00,3
7,2025-01-01,Airlift Winter Warm Hooded Runner - Espresso,Airlift Winter Warm Hooded Runner,Brown,Brown,W3735R040644,400.88,4
8,2025-01-01,Airmesh Venus Bralette - Champagne,Airmesh Venus Bralette,Champagne,Champagne,W9514R015072,48.00,1
9,2025-01-01,All Day Tank - White,All Day Tank,None,White,W2730R003,216.04,4


## 5. Did the derivation actually do anything?

Rows where `product` differs from `product_title` — i.e. a colour suffix was stripped.

In [7]:
changed = sample[sample["product"] != sample["product_title"].str.strip()]
print(f"{len(changed):,} of {len(sample):,} rows changed  ({len(changed) / len(sample):.1%})\n")

changed[["product_title", "product", "color", "resolved_color"]].drop_duplicates().head(15)

34,562 of 34,715 rows changed  (99.6%)



,product_title,product,color,resolved_color
0,"7"" Sports Club Sweater Knit Basketball Short -...","7"" Sports Club Sweater Knit Basketball Short",Black,Black
1,7/8 High-Waist Airbrush Legging - Classic Red,7/8 High-Waist Airbrush Legging,Classic Red,Classic Red
2,Accolade 1/4 Zip Pullover - Athletic Heather Grey,Accolade 1/4 Zip Pullover,Athletic Grey,Athletic Grey
3,Accolade Crew Neck Pullover - Celestial Blue,Accolade Crew Neck Pullover,Celestial Blue,Celestial Blue
4,Accolade Full Zip Hoodie - Black,Accolade Full Zip Hoodie,Black,Black
5,Accolade Hoodie - Athletic Heather Grey,Accolade Hoodie,None,Athletic Heather Grey
6,Accolade Hoodie - Bone,Accolade Hoodie,None,Bone
7,Airlift Winter Warm Hooded Runner - Espresso,Airlift Winter Warm Hooded Runner,Brown,Brown
8,Airmesh Venus Bralette - Champagne,Airmesh Venus Bralette,Champagne,Champagne
9,All Day Tank - White,All Day Tank,None,White


In [8]:
# Titles the derivation left alone - worth eyeballing for missed patterns
untouched = sample[sample["product"] == sample["product_title"].str.strip()]
untouched[["product_title", "product", "color"]].drop_duplicates().head(15)

,product_title,product,color
174,Sleep & Downshift Essential Oil Blend,Sleep & Downshift Essential Oil Blend,None
200,Head-To-Toe Glow Oil,Head-To-Toe Glow Oil,None
360,Glow System Discovery Set,Glow System Discovery Set,None
748,Daily SPF Face Moisturizer,Daily SPF Face Moisturizer,None
828,Uplift & Reset Essential Oil Blend,Uplift & Reset Essential Oil Blend,None
1210,Hydrate & Glow 1-Minute Mask,Hydrate & Glow 1-Minute Mask,None
1600,Magic Multi-Balm,Magic Multi-Balm,None
1604,Mini Dry Shampoo,Mini Dry Shampoo,None
1798,Lasting Lip Balm,Lasting Lip Balm,None
2029,Total Refresh Mat Spray,Total Refresh Mat Spray,None


## 6. First look at product distribution

In [9]:
print(f"distinct product_title : {sample['product_title'].nunique():,}")
print(f"distinct product       : {sample['product'].nunique():,}")
print(f"distinct sku           : {sample['sku'].nunique():,}")
print(f"collapse ratio         : {sample['product_title'].nunique() / max(sample['product'].nunique(), 1):.2f}x titles per product")

distinct product_title : 5,093
distinct product       : 1,418
distinct sku           : 15,181
collapse ratio         : 3.59x titles per product


In [10]:
# Top products in the sample by revenue, with colour spread
(
    sample.groupby("product")
    .agg(
        revenue=("revenue", "sum"),
        units=("ordered_quantities", "sum"),
        colors=("resolved_color", "nunique"),
        skus=("sku", "nunique"),
        first_date=("order_date", "min"),
        last_date=("order_date", "max"),
    )
    .sort_values("revenue", ascending=False)
    .head(20)
)

,revenue,units,colors,skus,first_date,last_date
product,,,,,,
Accolade Crew Neck Pullover,1874350.59,16529,44,211,2025-01-01,2026-09-12
Accolade Straight Leg Sweatpant,1643023.60,15586,30,155,2025-01-03,2026-09-15
Suit Up Trouser (Regular),1430363.41,12789,11,36,2025-01-01,2026-09-15
Accolade 1/4 Zip Pullover,795081.56,6655,13,62,2025-01-01,2026-09-14
7/8 High-Waist Airlift Legging,604874.80,5487,53,183,2025-01-02,2026-09-14
Accolade Sweatpant,565762.42,5059,25,127,2025-01-02,2026-09-15
Accolade Hoodie,469011.68,3835,21,100,2025-01-01,2026-09-14
ALO Runner,385944.53,2486,21,176,2025-07-24,2026-09-08
Match Point Tennis Skirt,374412.98,6030,17,53,2025-01-18,2026-09-11


In [11]:
# Daily shape of the sample
daily = (
    sample.groupby("order_date")
    .agg(rows=("sku", "size"), products=("product", "nunique"), revenue=("revenue", "sum"))
    .sort_index()
)
print(daily.describe().to_string())
daily.head(10)

             rows    products       revenue
count  623.000000  623.000000  6.230000e+02
mean    55.722311   50.481541  4.601905e+04
std     15.545130   13.415062  9.058568e+04
min     22.000000   20.000000  4.625000e+03
25%     44.000000   40.000000  1.951790e+04
50%     55.000000   50.000000  2.710156e+04
75%     65.500000   59.000000  3.958304e+04
max    134.000000  113.000000  1.234923e+06


,rows,products,revenue
order_date,,,
2025-01-01,41,40,18835.06
2025-01-02,31,30,13407.73
2025-01-03,26,26,10715.41
2025-01-04,40,37,25027.85
2025-01-05,44,36,17668.50
2025-01-06,33,33,16082.15
2025-01-07,55,54,17673.38
2025-01-08,36,36,10697.95
2025-01-09,37,37,13416.28


## 7. Clean the colour values

`resolved_color` falls back to the text after `" - "` whenever the raw `color` column is null.
For beauty / wellness / rewards lines that text is a **quantity, not a colour** - `60 Pack`,
`8 oz`, `50ML`, `450 points` - which is where columns like `30 Pack | units` came from.

Filtering by "is it in the raw `color` vocabulary?" would be wrong: ~18 genuine colours
(`Grey Tiedye`, `Java Brown`, `Bordeaux`, `Classic Red/White`, `Light Grey Iridescent`, ...)
appear *only* in titles and would be lost. The junk has a distinct shape instead - a number
followed by a unit - so match that directly.

In [12]:
import re

# A number + a unit ("60 Pack", "8 oz", "50ML", "450 points"), or the literal "All Colors".
NON_COLOR_RE = re.compile(r"^\s*(?:\d+\s*(?:pack|oz|ml|g|points?)|all colors)\s*$", re.IGNORECASE)

is_non_color = sample["resolved_color"].fillna("").map(lambda v: bool(NON_COLOR_RE.match(v)))
sample["color_clean"] = sample["resolved_color"].where(~is_non_color)

dropped = sample.loc[is_non_color, "resolved_color"]
print(f"excluded {is_non_color.sum():,} rows / {dropped.nunique()} distinct non-colour values")
print(f"revenue in excluded rows: ${sample.loc[is_non_color, 'revenue'].sum():,.2f} "
      f"(still counted in total_sales)")
print(f"\ncolours: {sample['resolved_color'].nunique()} -> {sample['color_clean'].nunique()}")
print(f"\ndropped values: {sorted(dropped.unique())}")

excluded 31 rows / 9 distinct non-colour values
revenue in excluded rows: $3,452.80 (still counted in total_sales)

colours: 434 -> 425

dropped values: ['10 Pack', '2 oz', '30 Pack', '450 points', '550 points', '60 Pack', '60 oz', '8 oz', 'All Colors']


## 8. Product x colour pivot (wide)

One row per **product**. `total_sales` is the product's total revenue across *all* rows;
then four columns per colour - `active_days` (distinct dates with a recorded sale),
`first_date`, `last_date`, `units`.

Written to `01_product_color_pivot.csv` for download.

In [13]:
METRIC_ORDER = ["active_days", "first_date", "last_date", "units"]

grouped = (
    sample.dropna(subset=["color_clean"])
    .groupby(["product", "color_clean"])
    .agg(
        active_days=("order_date", "nunique"),
        first_date=("order_date", "min"),
        last_date=("order_date", "max"),
        units=("ordered_quantities", "sum"),
    )
)

wide = grouped.unstack("color_clean")
wide.columns = wide.columns.set_names(["metric", "color"])
wide = wide.reorder_levels(["color", "metric"], axis=1)
wide = wide.reindex(
    columns=pd.MultiIndex.from_product(
        [sorted(grouped.index.get_level_values("color_clean").unique()), METRIC_ORDER],
        names=["color", "metric"],
    )
)
wide.columns = [f"{color} | {metric}" for color, metric in wide.columns]

total_sales = sample.groupby("product")["revenue"].sum().rename("total_sales")
pivot = total_sales.to_frame().join(wide).sort_values("total_sales", ascending=False)

pivot.to_csv("01_product_color_pivot.csv")

n_colors = (pivot.shape[1] - 1) // len(METRIC_ORDER)
print(f"{pivot.shape[0]:,} products x {pivot.shape[1]:,} columns "
      f"({n_colors:,} colours x {len(METRIC_ORDER)} metrics + total_sales)")
pivot.head(10)

1,418 products x 1,701 columns (425 colours x 4 metrics + total_sales)


,total_sales,Alo Scent | active_days,Alo Scent | first_date,Alo Scent | last_date,Alo Scent | units,Alpine Cocoa | active_days,Alpine Cocoa | first_date,Alpine Cocoa | last_date,Alpine Cocoa | units,Alpine Cocoa Heather | active_days,Alpine Cocoa Heather | first_date,Alpine Cocoa Heather | last_date,Alpine Cocoa Heather | units,Alpine Cocoa Suede | active_days,Alpine Cocoa Suede | first_date,Alpine Cocoa Suede | last_date,Alpine Cocoa Suede | units,Anthracite | active_days,Anthracite | first_date,Anthracite | last_date,Anthracite | units,Anthracite/Black | active_days,Anthracite/Black | first_date,Anthracite/Black | last_date,Anthracite/Black | units,...,Winter Frost/Navy/Anthracite | units,Winter Frost/White | active_days,Winter Frost/White | first_date,Winter Frost/White | last_date,Winter Frost/White | units,Winter Ivy | active_days,Winter Ivy | first_date,Winter Ivy | last_date,Winter Ivy | units,Winter Latte | active_days,Winter Latte | first_date,Winter Latte | last_date,Winter Latte | units,Woodland Tan | active_days,Woodland Tan | first_date,Woodland Tan | last_date,Woodland Tan | units,Woodrose | active_days,Woodrose | first_date,Woodrose | last_date,Woodrose | units,Woodrose Wash | active_days,Woodrose Wash | first_date,Woodrose Wash | last_date,Woodrose Wash | units
product,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Accolade Crew Neck Pullover,1874350.59,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,23.0,2025-01-28,2026-05-26,1047.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Accolade Straight Leg Sweatpant,1643023.60,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,16.0,2025-12-27,2026-08-07,284.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,17.0,2025-01-25,2026-09-02,237.0,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Suit Up Trouser (Regular),1430363.41,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,2.0,2025-12-21,2026-03-09,57.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Accolade 1/4 Zip Pullover,795081.56,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,19.0,2025-02-04,2026-05-13,134.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,2.0,2025-04-04,2025-10-07,3.0,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
7/8 High-Waist Airlift Legging,604874.80,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,26.0,2025-02-09,2026-08-23,117.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,5.0,2025-01-07,2026-05-06,31.0,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Accolade Sweatpant,565762.42,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,35.0,2025-01-28,2026-08-13,154.0,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Accolade Hoodie,469011.68,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,4.0,2025-04-06,2025-06-26,8.0,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
ALO Runner,385944.53,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
Match Point Tennis Skirt,374412.98,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,...,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN,NaN,NaT,NaT,NaN
